In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("../data/imdb_features.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (7919, 72)


,name,year,duration,genre,main_genre,rating,votes,director,actor_1,actor_2,...,genre_horror.1,genre_music.1,genre_musical.1,genre_mystery.1,genre_romance.1,genre_sci-fi.1,genre_sport.1,genre_thriller.1,genre_unknown,genre_war.1
0,#Gadhvi (He thought he was Gandhi),2019.0,109.0,Drama,Drama,7.0,8.0,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,...,0,0,0,0,0,0,0,0,0,0
1,#Yaaram,2019.0,110.0,"Comedy, Romance",Comedy,4.4,35.0,Ovais Khan,Prateik,Ishita Raj,...,0,0,0,0,0,0,0,0,0,0
2,...Aur Pyaar Ho Gaya,1997.0,147.0,"Comedy, Drama, Musical",Comedy,4.7,827.0,Rahul Rawail,Bobby Deol,Aishwarya Rai Bachchan,...,0,0,0,0,0,0,0,0,0,0
3,...Yahaan,2005.0,142.0,"Drama, Romance, War",Drama,7.4,1086.0,Shoojit Sircar,Jimmy Sheirgill,Minissha Lamba,...,0,0,0,0,0,0,0,0,0,0
4,?: A Question Mark,2012.0,82.0,"Horror, Mystery, Thriller",Horror,5.6,326.0,Allyson Patel,Yash Dave,Muntazir Ahmad,...,1,0,0,0,0,0,0,0,0,0


In [3]:
rec_cols = [
    "name", "genre", "director",
    "actor_1", "actor_2", "actor_3",
    "rating", "votes", "year"
]

movies = df[rec_cols].copy()

display(movies.head())

,name,genre,director,actor_1,actor_2,actor_3,rating,votes,year
0,#Gadhvi (He thought he was Gandhi),Drama,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid,7.0,8.0,2019.0
1,#Yaaram,"Comedy, Romance",Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor,4.4,35.0,2019.0
2,...Aur Pyaar Ho Gaya,"Comedy, Drama, Musical",Rahul Rawail,Bobby Deol,Aishwarya Rai Bachchan,Shammi Kapoor,4.7,827.0,1997.0
3,...Yahaan,"Drama, Romance, War",Shoojit Sircar,Jimmy Sheirgill,Minissha Lamba,Yashpal Sharma,7.4,1086.0,2005.0
4,?: A Question Mark,"Horror, Mystery, Thriller",Allyson Patel,Yash Dave,Muntazir Ahmad,Kiran Bhatia,5.6,326.0,2012.0


In [4]:
text_cols = ["name", "genre", "director", "actor_1", "actor_2", "actor_3"]

for col in text_cols:
    movies[col] = movies[col].fillna("Unknown")
    movies[col] = movies[col].astype(str).str.lower().str.strip()

movies["rating"] = pd.to_numeric(movies["rating"], errors="coerce")
movies["votes"] = pd.to_numeric(movies["votes"], errors="coerce")
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")

display(movies.head())

,name,genre,director,actor_1,actor_2,actor_3,rating,votes,year
0,#gadhvi (he thought he was gandhi),drama,gaurav bakshi,rasika dugal,vivek ghamande,arvind jangid,7.0,8.0,2019.0
1,#yaaram,"comedy, romance",ovais khan,prateik,ishita raj,siddhant kapoor,4.4,35.0,2019.0
2,...aur pyaar ho gaya,"comedy, drama, musical",rahul rawail,bobby deol,aishwarya rai bachchan,shammi kapoor,4.7,827.0,1997.0
3,...yahaan,"drama, romance, war",shoojit sircar,jimmy sheirgill,minissha lamba,yashpal sharma,7.4,1086.0,2005.0
4,?: a question mark,"horror, mystery, thriller",allyson patel,yash dave,muntazir ahmad,kiran bhatia,5.6,326.0,2012.0


In [5]:
movies = movies.drop_duplicates(
    subset=["name", "year", "director"]
).reset_index(drop=True)

print("Shape after removing duplicates:", movies.shape)

Shape after removing duplicates: (7918, 9)


In [6]:
movies["year_clean"] = movies["year"].fillna(0).astype(int).astype(str)

movies["movie_id"] = (
    movies["name"].astype(str) + " (" + movies["year_clean"] + ")"
)

display(movies[["movie_id", "name", "year", "director", "rating"]].head())

,movie_id,name,year,director,rating
0,#gadhvi (he thought he was gandhi) (2019),#gadhvi (he thought he was gandhi),2019.0,gaurav bakshi,7.0
1,#yaaram (2019),#yaaram,2019.0,ovais khan,4.4
2,...aur pyaar ho gaya (1997),...aur pyaar ho gaya,1997.0,rahul rawail,4.7
3,...yahaan (2005),...yahaan,2005.0,shoojit sircar,7.4
4,?: a question mark (2012),?: a question mark,2012.0,allyson patel,5.6


In [7]:
movies["combined_features"] = (
    movies["genre"].fillna("") + " " +
    movies["director"].fillna("") + " " +
    movies["actor_1"].fillna("") + " " +
    movies["actor_2"].fillna("") + " " +
    movies["actor_3"].fillna("")
)

display(movies[["movie_id", "combined_features"]].head())

,movie_id,combined_features
0,#gadhvi (he thought he was gandhi) (2019),drama gaurav bakshi rasika dugal vivek ghamand...
1,#yaaram (2019),"comedy, romance ovais khan prateik ishita raj ..."
2,...aur pyaar ho gaya (1997),"comedy, drama, musical rahul rawail bobby deol..."
3,...yahaan (2005),"drama, romance, war shoojit sircar jimmy sheir..."
4,?: a question mark (2012),"horror, mystery, thriller allyson patel yash d..."


In [8]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(movies["combined_features"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (7918, 5000)


In [9]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine similarity matrix shape:", cosine_sim.shape)

Cosine similarity matrix shape: (7918, 7918)


In [10]:
movie_indices = pd.Series(
    movies.index,
    index=movies["movie_id"]
).drop_duplicates()

display(movie_indices.head())

movie_id
#gadhvi (he thought he was gandhi) (2019)    0
#yaaram (2019)                               1
...aur pyaar ho gaya (1997)                  2
...yahaan (2005)                             3
?: a question mark (2012)                    4
dtype: int64

In [11]:
def search_movie(keyword, top_n=10):
    keyword = keyword.lower().strip()

    results = movies[movies["name"].str.contains(keyword, case=False, na=False)]

    if results.empty:
        return f"No movies found for keyword: {keyword}"

    return results[
        ["movie_id", "name", "year", "genre", "director", "rating"]
    ].head(top_n).reset_index(drop=True)

In [12]:
search_movie("don")

,movie_id,name,year,genre,director,rating
0,1920 london (2016),1920 london,2016.0,"horror, mystery",dharmendra suresh desai,4.1
1,bhale donga (1989),bhale donga,1989.0,"action, comedy, drama",kodanda rami reddy a.,3.3
2,chhodon naa yaar (2007),chhodon naa yaar,2007.0,"horror, thriller",dilip sood,4.6
3,do ladke dono kadke (1979),do ladke dono kadke,1979.0,"comedy, crime",basu chatterjee,6.0
4,don (2006),don,2006.0,"action, crime, thriller",farhan akhtar,7.2
5,don (1978),don,1978.0,"action, crime, thriller",chandra barot,7.8
6,don 2 (2011),don 2,2011.0,"action, crime, thriller",farhan akhtar,7.1
7,don muthu swami (2008),don muthu swami,2008.0,"comedy, drama",ashim s. samanta,4.7
8,dongri ka raja (2016),dongri ka raja,2016.0,"action, drama, musical",hadi abrar,3.9
9,guest iin london (2017),guest iin london,2017.0,comedy,ashwani dhir,5.4


In [13]:
search_movie("dangal")

,movie_id,name,year,genre,director,rating
0,dangal (2016),dangal,2016.0,"action, biography, drama",nitesh tiwari,8.4


In [14]:
def recommend_movies_by_id(movie_id, top_n=5):
    movie_id = movie_id.strip()

    if movie_id not in movie_indices:
        return f"Movie '{movie_id}' not found. Use search_movie() to find the correct movie_id."

    idx = movie_indices[movie_id]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:top_n + 1]

    rec_indices = [i[0] for i in similarity_scores]
    scores = [i[1] for i in similarity_scores]

    recommendations = movies.iloc[rec_indices][
        ["movie_id", "name", "year", "genre", "director", "actor_1", "rating", "votes"]
    ].copy()

    recommendations["similarity_score"] = scores

    return recommendations.reset_index(drop=True)

In [15]:
don_results = search_movie("don")
display(don_results)

,movie_id,name,year,genre,director,rating
0,1920 london (2016),1920 london,2016.0,"horror, mystery",dharmendra suresh desai,4.1
1,bhale donga (1989),bhale donga,1989.0,"action, comedy, drama",kodanda rami reddy a.,3.3
2,chhodon naa yaar (2007),chhodon naa yaar,2007.0,"horror, thriller",dilip sood,4.6
3,do ladke dono kadke (1979),do ladke dono kadke,1979.0,"comedy, crime",basu chatterjee,6.0
4,don (2006),don,2006.0,"action, crime, thriller",farhan akhtar,7.2
5,don (1978),don,1978.0,"action, crime, thriller",chandra barot,7.8
6,don 2 (2011),don 2,2011.0,"action, crime, thriller",farhan akhtar,7.1
7,don muthu swami (2008),don muthu swami,2008.0,"comedy, drama",ashim s. samanta,4.7
8,dongri ka raja (2016),dongri ka raja,2016.0,"action, drama, musical",hadi abrar,3.9
9,guest iin london (2017),guest iin london,2017.0,comedy,ashwani dhir,5.4


In [19]:
recommend_movies_by_id("guest iin london (2017)	", top_n=5)

,movie_id,name,year,genre,director,actor_1,rating,votes,similarity_score
0,one two three (2008),one two three,2008.0,"comedy, crime",ashwani dhir,tusshar kapoor,5.3,2511.0,0.573682
1,atithi tum kab jaoge? (2010),atithi tum kab jaoge?,2010.0,comedy,ashwani dhir,ajay devgn,6.4,5383.0,0.546938
2,son of sardaar (2012),son of sardaar,2012.0,"action, comedy",ashwani dhir,anil devgan,4.0,9060.0,0.386723
3,luka chuppi (2019),luka chuppi,2019.0,"comedy, romance",laxman utekar,kartik aaryan,6.3,8808.0,0.349794
4,kaanchi (2014),kaanchi,2014.0,"drama, musical, romance",subhash ghai,mishti,4.2,590.0,0.284940


In [20]:
movies["rating_filled"] = movies["rating"].fillna(movies["rating"].mean())
movies["votes_filled"] = movies["votes"].fillna(movies["votes"].median())

movies["normalized_rating"] = (
    movies["rating_filled"] - movies["rating_filled"].min()
) / (
    movies["rating_filled"].max() - movies["rating_filled"].min()
)

movies["normalized_votes"] = (
    np.log1p(movies["votes_filled"]) - np.log1p(movies["votes_filled"]).min()
) / (
    np.log1p(movies["votes_filled"]).max() - np.log1p(movies["votes_filled"]).min()
)

display(movies[["movie_id", "rating", "votes", "normalized_rating", "normalized_votes"]].head())

,movie_id,rating,votes,normalized_rating,normalized_votes
0,#gadhvi (he thought he was gandhi) (2019),7.0,8.0,0.662921,0.035262
1,#yaaram (2019),4.4,35.0,0.370787,0.155825
2,...aur pyaar ho gaya (1997),4.7,827.0,0.404494,0.428512
3,...yahaan (2005),7.4,1086.0,0.707865,0.452181
4,?: a question mark (2012),5.6,326.0,0.505618,0.347714


In [21]:
def recommend_movies_hybrid(
    movie_id,
    top_n=5,
    similarity_weight=0.7,
    rating_weight=0.2,
    popularity_weight=0.1
):
    movie_id = movie_id.strip()

    if movie_id not in movie_indices:
        return f"Movie '{movie_id}' not found. Use search_movie() to find the correct movie_id."

    idx = movie_indices[movie_id]

    similarity_scores = np.array(cosine_sim[idx]).flatten()

    hybrid_scores = (
        similarity_weight * similarity_scores +
        rating_weight * movies["normalized_rating"].values +
        popularity_weight * movies["normalized_votes"].values
    )

    hybrid_scores[idx] = -1

    top_indices = hybrid_scores.argsort()[::-1][:top_n]

    recommendations = movies.iloc[top_indices][
        ["movie_id", "name", "year", "genre", "director", "actor_1", "rating", "votes"]
    ].copy()

    recommendations["similarity_score"] = similarity_scores[top_indices]
    recommendations["hybrid_score"] = hybrid_scores[top_indices]

    return recommendations.reset_index(drop=True)

In [22]:
recommend_movies_hybrid("don (2006)", top_n=5)

,movie_id,name,year,genre,director,actor_1,rating,votes,similarity_score,hybrid_score
0,don 2 (2011),don 2,2011.0,"action, crime, thriller",farhan akhtar,shah rukh khan,7.1,50435.0,0.638049,0.660056
1,the sky is pink (2019),the sky is pink,2019.0,"drama, romance",shonali bose,priyanka chopra jonas,7.6,7683.0,0.462662,0.532157
2,rock on!! (2008),rock on!!,2008.0,"drama, music",abhishek kapoor,arjun rampal,7.7,21178.0,0.437858,0.525859
3,dil dhadakne do (2015),dil dhadakne do,2015.0,"comedy, drama, romance",zoya akhtar,anil kapoor,6.9,15722.0,0.455702,0.517782
4,om shanti om (2007),om shanti om,2007.0,"action, comedy, drama",farah khan,shah rukh khan,6.7,40120.0,0.447769,0.515882


In [23]:
def recommend_by_keyword(keyword, top_n=5):
    results = search_movie(keyword, top_n=10)

    if isinstance(results, str):
        return results

    display(results)

    selected_movie_id = results.iloc[0]["movie_id"]

    print(f"Showing recommendations for: {selected_movie_id}")

    return recommend_movies_hybrid(selected_movie_id, top_n=top_n)

In [24]:
recommend_by_keyword("dangal", top_n=5)

,movie_id,name,year,genre,director,rating
0,dangal (2016),dangal,2016.0,"action, biography, drama",nitesh tiwari,8.4


Showing recommendations for: dangal (2016)


,movie_id,name,year,genre,director,actor_1,rating,votes,similarity_score,hybrid_score
0,suraj pe mangal bhari (2020),suraj pe mangal bhari,2020.0,comedy,abhishek sharma,manoj bajpayee,5.9,3011.0,0.436268,0.467334
1,chhichhore (2019),chhichhore,2019.0,"comedy, drama",nitesh tiwari,sushant singh rajput,8.3,38581.0,0.250233,0.413221
2,like stars on earth (2007),like stars on earth,2007.0,"drama, family",aamir khan,amole gupte,8.4,175810.0,0.195481,0.390332
3,children's party (2011),children's party,2011.0,"comedy, drama, family",vikas bahl,nitesh tiwari,7.5,7296.0,0.261393,0.388572
4,mohalla assi (2015),mohalla assi,2015.0,"comedy, drama",chandra prakash dwivedi,sunny deol,6.8,1601.0,0.298605,0.385704


In [25]:
recommend_by_keyword("sholay", top_n=5)

,movie_id,name,year,genre,director,rating
0,aag ke sholay (1988),aag ke sholay,1988.0,"action, thriller",s.r. pratap,3.4
1,do sholay (1977),do sholay,1977.0,action,sukhdev ahluwalia,3.3
2,duplicate sholay (2002),duplicate sholay,2002.0,comedy,kanti shah,6.5
3,ramgarh ke sholay (1991),ramgarh ke sholay,1991.0,"action, comedy, crime",ajit dewani,4.7
4,sholay (1975),sholay,1975.0,"action, adventure, comedy",ramesh sippy,8.2
5,the sholay girl (2019),the sholay girl,2019.0,"action, biography, drama",aditya sarpotdar,7.7


Showing recommendations for: aag ke sholay (1988)


,movie_id,name,year,genre,director,actor_1,rating,votes,similarity_score,hybrid_score
0,ab meri baari (1989),ab meri baari,1989.0,action,bhagwan thakur,hemant birje,5.1,8.0,0.561586,0.486524
1,chandaal atma (1999),chandaal atma,1999.0,horror,s.r. pratap,aparna,4.6,9.0,0.511727,0.441303
2,shirdi ke sai baba (1977),shirdi ke sai baba,1977.0,"biography, drama",ashok v. bhushan,sudhir dalvi,7.0,229.0,0.326499,0.392844
3,reshma aur sultaan (2002),reshma aur sultaan,2002.0,action,s. kumar,hemant birje,4.9,10.0,0.426325,0.389092
4,tarzan's daughter (2002),tarzan's daughter,2002.0,"action, adventure, musical",pratap singh,hemant birje,4.5,11.0,0.419017,0.375745


In [26]:
os.makedirs("../models", exist_ok=True)

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, "../models/tfidf_matrix.pkl")
joblib.dump(cosine_sim, "../models/cosine_similarity.pkl")
joblib.dump(movie_indices, "../models/movie_indices.pkl")

movies.to_csv("../models/recommendation_movies.csv", index=False)

print("Recommendation system objects saved successfully!")

Recommendation system objects saved successfully!


In [27]:
print("=" * 70)
print("RECOMMENDATION SYSTEM SUMMARY")
print("=" * 70)

print("Recommendation Type: Content-Based + Hybrid Ranking")
print("Technique Used: TF-IDF + Cosine Similarity")
print("\nFeatures Used:")
print("• Genre")
print("• Director")
print("• Actor 1")
print("• Actor 2")
print("• Actor 3")
print("• Rating")
print("• Votes")

print("\nOutput Includes:")
print("• Similar movies")
print("• Similarity score")
print("• Hybrid score")
print("• Movie rating")
print("• Popularity through votes")

print("\nNotebook 6 completed successfully!")

RECOMMENDATION SYSTEM SUMMARY
Recommendation Type: Content-Based + Hybrid Ranking
Technique Used: TF-IDF + Cosine Similarity

Features Used:
• Genre
• Director
• Actor 1
• Actor 2
• Actor 3
• Rating
• Votes

Output Includes:
• Similar movies
• Similarity score
• Hybrid score
• Movie rating
• Popularity through votes

Notebook 6 completed successfully!
